In [ ]:
model_type_str = "izumil_jppost-cp-e1-b512_jsts-100-cp-e3-b1"
model_name_str = "../../fine-tuning/tuned_models/izumil_jppost-cp-e1-b512_jsts-100-cp-e1to5-b1/checkpoints/37353"
task_str = "pooling"
quantize_bool = False
opset_int = 12

print(f"{model_name_str = }")
print(f"{model_type_str = }")
print(f"{task_str = }")
print(f"{quantize_bool = }")
print(f"{opset_int = }")

In [ ]:
task_type_str = ""
if task_str == "default":
    task_type_str = "df"
elif task_str == "pooling":
    task_type_str = "pl"
elif task_str == "text-classification":
    task_type_str = "cl"
elif task_str == "question-answering":
    task_type_str = "qa"

quantize_type_str = ""
if quantize_bool == True:
    quantize_type_str = "-qT"
elif quantize_bool == False:
    quantize_type_str = "-qF"

opset_type_str = "-op" + str(opset_int // 10) + str(opset_int % 10)

modeltype_output_str = model_type_str + "_onnx-" + task_type_str + quantize_type_str + opset_type_str
modelpath_output_str = "onnxed_models/" + modeltype_output_str

print(f"{modelpath_output_str = }")

In [ ]:
from txtai.pipeline import HFOnnx, Labels

onnx = HFOnnx()
onnxed_model = onnx(
    path=model_name_str,
    task=task_str,
    output=modelpath_output_str,
    quantize=quantize_bool,
    opset=opset_int
)

In [ ]:
import json

filepath_dataset_eval_str = "../../datasets/jsts-v1.1/valid-v1.1.json"

label_str = "label"
sentence1_str = "sentence1"
sentence2_str = "sentence2"


data_eval_dict = dict()
labels_list = list()
sentences1_list = list()
sentences2_list = list()

with open(filepath_dataset_eval_str) as fp:
    for line in fp:
        line_dict = json.loads(line)
        labels_list.append(line_dict[label_str])
        sentences1_list.append(line_dict[sentence1_str])
        sentences2_list.append(line_dict[sentence2_str])

data_eval_dict[label_str] = labels_list
data_eval_dict[sentence1_str] = sentences1_list
data_eval_dict[sentence2_str] = sentences2_list

# print(data_eval_dict)

In [ ]:
from onnxruntime import InferenceSession, SessionOptions
from sentence_transformers import util
from scipy.stats import pearsonr, spearmanr
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(model_name_str)
tokens1_list = tokenizer(data_eval_dict[sentence1_str], padding=True, truncation=True, return_tensors='np')
tokens2_list = tokenizer(data_eval_dict[sentence2_str], padding=True, truncation=True, return_tensors='np')


# Start inference session
options = SessionOptions()
session = InferenceSession(onnxed_model, options, providers = ["CPUExecutionProvider"])


embeddings1_list = session.run(None, dict(tokens1_list))[0]
embeddings2_list = session.run(None, dict(tokens2_list))[0]


cossim_list = list()
for embedding1, embedding2 in zip(embeddings1_list, embeddings2_list):
    cossim_float = util.cos_sim(embedding1, embedding2).tolist()[0][0]
    cossim_list.append(cossim_float)


pearson_cossim_float, _ = pearsonr(data_eval_dict[label_str], cossim_list)
spearman_cossim_float, _ = spearmanr(data_eval_dict[label_str], cossim_list)


print(f"{pearson_cossim_float = }")
print(f"{spearman_cossim_float = }")


filepath_scores_str = "scores/scores_" + modeltype_output_str + ".txt"

with open(filepath_scores_str, "w") as fp:
    fp.write(f"pearson_cossim: {pearson_cossim_float}\n")
    fp.write(f"spearman_cossim: {spearman_cossim_float}\n")

In [ ]:
import pandas as pd

filepath_embeddings_str = "embeddings/embeddings_" + modeltype_output_str + ".csv"

index_list = data_eval_dict[sentence1_str] + data_eval_dict[sentence2_str]
embeddings_list = embeddings1_list.tolist() + embeddings2_list.tolist()

embeddings_sr = pd.Series(embeddings_list, index=index_list)

embeddings_sr.to_csv(filepath_embeddings_str)

In [ ]:
import time
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

%matplotlib inline
# plt.style.use('dark_background')
plt.style.use('default')


### Choose a dataset
dataset_name = 'mixed'
# path_dataset_str = '../../datasets/jppost/mixed/data_sbert_speed_900.csv'
# path_dataset_str = '../../datasets/jppost/mixed/data_sbert_speed_700.csv'
path_dataset_str = '../../datasets/jppost/mixed/data_sbert_speed_200.csv'


y_max_float = 0.020


column_charslen_str = 'the_num_of_chars'
column_tokenslen_str = 'the_num_of_tokens'
column_elapsedtime_str = 'elapsed_time'    


def tokenize(model, sentences_str_list):
    sentences_tokenized = tokenizer(sentences_str_list, padding=True, truncation=True, return_tensors='pt')
    tokens_str_list = tokenizer.convert_ids_to_tokens(sentences_tokenized['input_ids'][0])

    return tokens_str_list


def get_elapsed_times(model, data_df):
    column_sentence_int = 1
    # column_sentence_str = 'question'
    
    data_df[column_tokenslen_str] = 0
    data_df[column_elapsedtime_str] = 0.
    
    # session.run(None, dict(tokens1_list))[0]
    
    counter = 0
    for row_pdtuple in data_df.itertuples():
        row_index_int = row_pdtuple.Index
        
        sentence_str = row_pdtuple[column_sentence_int]
        tokens_list = tokenize(model, sentence_str)
        data_df.loc[row_index_int, column_tokenslen_str] = len(tokens_list)
        print(len(tokens_list))
        
        # tokenized_list = tokenizer(
        #     row_pdtuple[column_sentence_int], padding=True, truncation=True, max_length=128, return_tensors='np'
        # )
        
        start_time = time.time()
        tokenized_list = tokenizer(row_pdtuple[column_sentence_int], padding=True, truncation=True, max_length=128, return_tensors='np')
        embeddings_list = session.run(None, dict(tokenized_list))[0]
        end_time = time.time()
        
        elapsed_time = end_time - start_time
        print(counter, elapsed_time)
        counter += 1
        data_df.loc[row_index_int, column_elapsedtime_str] = elapsed_time
    
    return data_df


def draw_graph(data_df, column_str, color_plot_str='#086972', color_lr_str='#DF5E88'):
    # Set coordinates
    x_max_float = 0.0
    if column_str == column_charslen_str:
        x_max_float = 920.0
    elif column_str == column_tokenslen_str:
        x_max_float = 530.0
    
    x_describe_df = data_df[column_str].describe()
    y_describe_df = data_df[column_elapsedtime_str].describe()
    x_min_float = x_max_float / 36
    x_mid_float = x_max_float / 2
    x_indent_float = x_max_float / 18
    y_incrm_float = y_max_float / 30
    y_title_float = y_max_float - y_incrm_float
    y_stats_float = y_max_float / 3
    y_lr_float = y_max_float / 2
    
    
    # Format data
    x_sr = data_df[column_str].values
    x_df = data_df[[column_str]].values
    y_sr = data_df[column_elapsedtime_str].values
    
    # Linear Regression
    lr = LinearRegression()
    lr.fit(x_df, y_sr)
    
    coef_float = lr.coef_[0]
    intercept_float = lr.intercept_
    
    # Plot
    plt.clf() # Clear the plot
    elapsed_snsplot = sns.jointplot(x=x_sr, y=y_sr, xlim=(0, x_max_float), ylim=(0, y_max_float), color=color_plot_str)
    plt.plot(x_df, lr.predict(x_df), color=color_lr_str)
    
    # Print titles
    plt.text(x_min_float, y_title_float, f'#{dataset_name.upper()}', horizontalalignment='left')
    plt.text(x_min_float, y_title_float - y_incrm_float, f'#{modeltype_output_str}', horizontalalignment='left')
    
    # Print the coefficient and the intercept of Linear Regression
    plt.text(x_mid_float, y_lr_float, '<Linear Regression>', horizontalalignment='left')
    plt.text(x_mid_float + x_indent_float, y_lr_float - y_incrm_float * 1, 'coef', horizontalalignment='left')
    plt.text(x_max_float, y_lr_float - y_incrm_float * 1, '%.8f' % coef_float, horizontalalignment='right')
    plt.text(x_mid_float + x_indent_float, y_lr_float - y_incrm_float * 2, 'intercept', horizontalalignment='left')
    plt.text(x_max_float, y_lr_float - y_incrm_float * 2, '%.8f' % intercept_float, horizontalalignment='right')
        
    # Print stats of x
    plt.text(x_min_float, y_stats_float + y_incrm_float * 0.5, f'<{column_str}>', horizontalalignment='left')
    
    for i, key in enumerate(list(x_describe_df.index)):
        plt.text(x_min_float + x_indent_float, y_stats_float - y_incrm_float * (i + 1), key, horizontalalignment='left')
        plt.text(x_mid_float, y_stats_float - y_incrm_float * (i + 1), '%.8f' % x_describe_df[key], horizontalalignment='right')
        
    # Print stats of y
    plt.text(x_mid_float, y_stats_float + y_incrm_float * 0.5, f'<{column_elapsedtime_str}>', horizontalalignment='left')
    
    for i, key in enumerate(list(x_describe_df.index)):
        plt.text(x_mid_float + x_indent_float, y_stats_float - y_incrm_float * (i + 1), key, horizontalalignment='left')
        plt.text(x_max_float, y_stats_float - y_incrm_float * (i + 1), '%.8f' % y_describe_df[key], horizontalalignment='right')
        

    plt.show()
    
    # Save the plot
    x_type_str = column_str.replace('the_num_of', '')
    # filepath_output = 'elapsed_time/elapsed_time_' + model_type_str + x_type_str + '_' + dataset_name + '_nt.png'
    filepath_output = 'elapsed_time/elapsed_time_' + modeltype_output_str + x_type_str + '_' + dataset_name + '_h' + str(y_max_float) + '.png'
    # filepath_output = 'elapsed_time/elapsed_time_' + modelpath_type_str + x_type_str + '_' + dataset_name + '900.png'
    elapsed_snsplot.figure.savefig(filepath_output)



In [ ]:
# Load the datasets
data_df = pd.read_csv(path_dataset_str, index_col=0)
# data_df

# Measure elapsed time for encoding each sentence in the datasets
data_df = get_elapsed_times(onnxed_model, data_df)
# data_df

# Draw the graphs
draw_graph(data_df, column_charslen_str, color_plot_str='#086972')
draw_graph(data_df, column_tokenslen_str, color_plot_str='#277BC0')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sentence_transformers import models, SentenceTransformer, util

%matplotlib inline
# plt.style.use('dark_background')
plt.style.use('default')

COSSIM_NAME = 'cos_sim'


def set_chimei_categories_list():
    chimei_categories_list = list()
    chimei_categories_list.append('00_prefectures')
    chimei_categories_list.append('01_shi')
    chimei_categories_list.append('02_ku')
    chimei_categories_list.append('03_gun')
    chimei_categories_list.append('04_cho')
    chimei_categories_list.append('05_son')
    
    return chimei_categories_list


def set_filepaths_input_dict(chimei_categories_list):
    filepaths_input_base = '../../datasets/jppost/local_govt_lists/'
    filepaths_input_dict = dict()

    for chimei_category in chimei_categories_list:
        filepaths_input = filepaths_input_base + chimei_category + '_list.txt'
        filepaths_input_dict[chimei_category] = filepaths_input
    
    return filepaths_input_dict
        

def set_chimeis_lists_dict(filepaths_input_dict):
    chimeis_lists_dict = dict()
    
    for chimei_category, filepath_input in filepaths_input_dict.items():
        chimeis_list = list()
        
        with open(filepath_input) as fp:
            for line in fp:
                chimeis_list.append(line.rstrip())
        
        chimeis_lists_dict[chimei_category] = chimeis_list
    
    return chimeis_lists_dict


def embed_chimei(model, chimeis_lists_dict):
    embeddings_dicts_dict = dict()
    
    for chimei_category, chimeis_list in chimeis_lists_dict.items():
        embeddings_dict = dict()
        
        for chimei in chimeis_list:
            chimei_tokenized = tokenizer([chimei], padding=True, truncation=True, max_length=128, return_tensors='np')
            chimei_embedding = session.run(None, dict(chimei_tokenized))[0]
            embeddings_dict[chimei] = chimei_embedding
        
        embeddings_dicts_dict[chimei_category] = embeddings_dict
    
    return embeddings_dicts_dict


def set_filepaths_output_dict(chimei_categories_list, directory, prefix, ext):
    filepaths_output_dict = dict()

    for chimei_category in chimei_categories_list:
        filepaths_output = directory + prefix + modeltype_output_str + '_' + chimei_category + ext
        filepaths_output_dict[chimei_category] = filepaths_output
    
    return filepaths_output_dict


def create_cossim_matrix_dfs_dict(embeddings_dicts_dict):
    cossim_matrix_dfs_dict = dict()
    covered_chimeis_list = list()
    
    for chimei_category, embeddings_dict in embeddings_dicts_dict.items():
        chimeis_list = list(embeddings_dict.keys())
        cossim_matrix_df = pd.DataFrame(index=chimeis_list, columns=chimeis_list)
        
        for column in chimeis_list:
            covered_chimeis_list.append(column)
            
            for row in chimeis_list:
                if row not in covered_chimeis_list:
                    cossim_tensor = util.cos_sim(embeddings_dict[row], embeddings_dict[column])[0][0]
                    cossim_matrix_df.loc[row, column] = cossim_tensor.tolist()
            
            cossim_matrix_df[column] = pd.to_numeric(cossim_matrix_df[column], errors='coerce')
        
        cossim_matrix_df = cossim_matrix_df.transpose()
        cossim_matrix_dfs_dict[chimei_category] = cossim_matrix_df
        
    return cossim_matrix_dfs_dict


def melt_cossim_dfs_dict(cossim_matrix_dfs_dict):
    cossim_melted_dfs_dict = dict()
    
    for chimei_category, cossim_matrix_df in cossim_matrix_dfs_dict.items():
        
        cossim_melted_df = pd.melt(
            cossim_matrix_df.reset_index(),
            id_vars='index',
            var_name='local_govt_name2',
            value_name=COSSIM_NAME
        )
        cossim_melted_df = cossim_melted_df.rename(columns={'index': 'local_govt_name1'})
        
        cossim_melted_df = cossim_melted_df.dropna()
        
        # Sort by cosine similarity and reset the index
        cossim_melted_df = cossim_melted_df.sort_values(COSSIM_NAME, ascending=False)
        cossim_melted_df = cossim_melted_df.reset_index()
        cossim_melted_df = cossim_melted_df.drop(columns='index')
        
        cossim_melted_dfs_dict[chimei_category] = cossim_melted_df
        
    return cossim_melted_dfs_dict


def plot_and_save_cossim_hists(cossim_dfs_dict, filepaths_output_cossim_hist_dict):
    
    for chimei_category, cossim_df in cossim_dfs_dict.items():
        plt.clf() # Clear the plot
        cossim_hist_plot = sns.histplot(cossim_df[COSSIM_NAME], kde=True, binrange=(-1,1))
        
        # Get the maximun y-coordinate
        ys_np = cossim_hist_plot.lines[0].get_ydata()
        y_max_float = np.max(ys_np)
        
        # Get the stats
        cossim_describe_df = cossim_df[COSSIM_NAME].describe()
        
        # Add the stats table on the plot
        for i, key in enumerate(list(cossim_describe_df.index)):
            plt.text(-0.99, y_max_float - i * y_max_float / 16, key, horizontalalignment='left')
            plt.text(-0.19, y_max_float - i * y_max_float / 16, '%.8f' % cossim_describe_df[key], horizontalalignment='right')
        
        # Draw lines for the stats
        color_quarters = 'orange'
        color_standard = 'gray'
        linewidth_float = 0.5
        zorder_int = -1
        
        std_lower_float = cossim_describe_df['mean'] - cossim_describe_df['std']
        std_upper_float = cossim_describe_df['mean'] + cossim_describe_df['std']
        
        plt.axvline(cossim_describe_df['max'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['min'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['50%'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['25%'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['75%'], color=color_quarters, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(cossim_describe_df['mean'], color=color_standard, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(std_lower_float, color=color_standard, linewidth=linewidth_float, zorder=zorder_int)
        plt.axvline(std_upper_float, color=color_standard, linewidth=linewidth_float, zorder=zorder_int)

        
        fig_hist = cossim_hist_plot.get_figure()
        fig_hist.savefig(filepaths_output_cossim_hist_dict[chimei_category])

    return None


In [ ]:
# Create a chimei category list.
chimei_categories_list = set_chimei_categories_list()

# Create a dictionary of the paths of the input files by chimei category.
filepaths_input_dict = set_filepaths_input_dict(chimei_categories_list)

# Create a dictionary of chimei lists by chimei category.
chimeis_lists_dict = set_chimeis_lists_dict(filepaths_input_dict)

# Embed chimei lists via the model.
embeddings_dicts_dict = embed_chimei(onnxed_model, chimeis_lists_dict)

# Create a dictionary of the dataframes whose values are the cosine similarities of the chimei pairs by chimei category.
cossim_matrix_dfs_dict = create_cossim_matrix_dfs_dict(embeddings_dicts_dict)

# Create a dictionary of the dataframes created with cossim matrices melted by chimei category.
cossim_melted_dfs_dict = melt_cossim_dfs_dict(cossim_matrix_dfs_dict)

# Create a dictionary of the paths of the files to save the histgrams of the melted cossim matrices as by chimei category.
filepaths_output_cossim_mltd_hist_dict = set_filepaths_output_dict(
    chimei_categories_list=chimei_categories_list,
    directory='./cossim_1_melted_1_hist/',
    prefix='cossim_melted_hist2_',
    ext='.png'
)
# Plot and save histgrams of melted cossim matrices as files by chimei category.
plot_and_save_cossim_hists(cossim_melted_dfs_dict, filepaths_output_cossim_mltd_hist_dict)
